## SEttingup enviroment


In [1]:
import pandas as pd
import numpy as np


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 3, Finished, Available, Finished, False)

# Setting PArams

 ## SEtting CM and LM

In [2]:
BatchLM = "Apr'26"
BatchCM = "May'26"


BatchLM_Current_Bill="Apr-2026"
BatchLM_Next_Bill="May-2026"

BatchCM_Current_Bill="May-2026"
BatchCM_Next_Bill="Jun-2026"



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 4, Finished, Available, Finished, False)

## Setting Date and Times

In [3]:

from datetime import datetime


LM_Start_Date = pd.to_datetime(datetime(2026, 3, 27)).normalize()
LM_End_Date   = pd.to_datetime(datetime(2026, 4, 27)).normalize()
CM_Start_Date = pd.to_datetime(datetime(2026, 4, 28)).normalize()
CM_End_Date   = pd.to_datetime(datetime(2026, 5, 30)).normalize()



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 5, Finished, Available, Finished, False)

# Importing Date

## Mouting Drive

In [4]:
!!pip install openpyxl


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 6, Finished, Available, Finished, False)

['Requirement already satisfied: openpyxl in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (3.0.10)',
 'Requirement already satisfied: et_xmlfile in /home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages (from openpyxl) (1.1.0)']

## Input Files readiness Check

In [5]:
# List files inside Inputs folder in Lakehouse

files = mssparkutils.fs.ls("Files")

file_names = [file.name for file in files]



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 7, Finished, Available, Finished, False)

## Importing Bill1 Schedule

In [6]:
import pandas as pd


# Define Lakehouse file path
file_path = "/lakehouse/default/Files/Bill1_Schedule.xlsx"

# Read specific sheet
Bill1_Schedule = pd.read_excel(
    file_path,
    sheet_name="BILL1_Schedule",
    engine='openpyxl'
)

# Preview
Bill1_Schedule.head()

# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)



Bill1_Schedule.columns = Bill1_Schedule.columns.str.replace("\n", " ", regex=True)
Bill1_Schedule.columns = Bill1_Schedule.columns.str.strip()
Bill1_Schedule.columns = Bill1_Schedule.columns.str.replace(" +", " ", regex=True)
Bill1_Schedule = Bill1_Schedule.rename(columns={"Cycle Day's": "CD"})
Bill1_Schedule = Bill1_Schedule.rename(columns={"Due Date": "Bill1_Sch_Due_Date"})


# Convert CD to integer
Bill1_Schedule["CD"] = Bill1_Schedule["CD"].astype("Int64")

# Convert Bill1_Sch_Due_Date to date
Bill1_Schedule["Bill1_Sch_Due_Date"] = pd.to_datetime(
    Bill1_Schedule["Bill1_Sch_Due_Date"], errors="coerce"
).dt.normalize()

# Convert Issue Date to date
Bill1_Schedule["Issue Date"] = pd.to_datetime(
    Bill1_Schedule["Issue Date"], errors="coerce"
).dt.normalize()



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 8, Finished, Available, Finished, False)

## Importing Bill2 Schedule

In [7]:


# Define Lakehouse path
file_path = "/lakehouse/default/Files/Bill2_Schedule.xlsx"

# Read specific sheet
Bill2_Schedule = pd.read_excel(
    file_path,
    sheet_name="Bill2_Schedule",
    engine='openpyxl'
)

# Preview
Bill2_Schedule.head()
# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)
Bill2_Schedule = Bill2_Schedule.rename(columns={"Cycle \nDay's": "CD"})


# Convert CD to integer
Bill2_Schedule["CD"] = Bill2_Schedule["CD"].astype("Int64")



# Convert Issue Date to date
Bill2_Schedule["Issue Date"] = pd.to_datetime(
    Bill2_Schedule["Issue Date"], errors="coerce"
).dt.normalize()




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 9, Finished, Available, Finished, False)

## Importing LST

In [8]:

# Define Lakehouse path
file_path = "/lakehouse/default/Files/LST.xlsx"

# Read specific sheet
LST = pd.read_excel(
    file_path,
    sheet_name="Data",
    engine='openpyxl'
)


# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 10, Finished, Available, Finished, False)

### LST Trimming

In [9]:
# Trimming LST
# Step 1: copy Email Date to Revoaktion
LST["Revoaktion"] = LST["Email Date"]

# Keep only required columns
LST = LST[[
    "Contract",
    "Email Date",
    "Revoaktion",
    "Assigned Agency",
    "Cycle Day",
    "Due Date",
    "IBC Code"
]]

# --------------------------
# Convert data types
# --------------------------


# ---- Conversions on LST ----
# Convert to Integer (nullable Int64 to keep NaNs)
LST["Contract"]   = pd.to_numeric(LST["Contract"], errors="coerce").astype("Int64")
LST["IBC Code"]   = pd.to_numeric(LST["IBC Code"], errors="coerce").astype("Int64")
LST["Cycle Day"]  = pd.to_numeric(LST["Cycle Day"], errors="coerce").astype("Int64")

# Convert to datetime and then normalize to remove time component
LST["Email Date"] = pd.to_datetime(LST["Email Date"], errors="coerce").dt.normalize()
LST["Due Date"]   = pd.to_datetime(LST["Due Date"], errors="coerce").dt.normalize()
LST["Revoaktion"] = (
    pd.to_datetime(LST["Email Date"], errors="coerce")  # convert to datetime
    .dt.normalize()                                     # set time to 00:00
    + pd.Timedelta(days=24)                             # add 24 days
)

# Keep as text
LST["Assigned Agency"] = LST["Assigned Agency"].astype("string")

# ---- Rename column ----
# Rename "Email Date" to "Assignment Data" (as you wrote)
LST = LST.rename(columns={"Email Date": "Assignment Date"})

LST = LST.rename(columns={"Cycle Day": "CD"})



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 11, Finished, Available, Finished, False)

### Assigning Batch names

In [10]:
 #if pd.isna(x):
       # return "Other"

# 3) Simple IF logic
def find_batch(x):

    if LM_Start_Date <= x <= LM_End_Date:
        return BatchLM
    if CM_Start_Date <= x <= CM_End_Date:
        return BatchCM
    return "Other"

LST["Batch"] = LST["Assignment Date"].apply(find_batch)





StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 12, Finished, Available, Finished, False)

In [11]:
print("LM:", LM_Start_Date, LM_End_Date)
print("CM:", CM_Start_Date, CM_End_Date)
print("Row date:", LST["Assignment Date"].iloc[0])

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 13, Finished, Available, Finished, False)

LM: 2026-03-27 00:00:00 2026-04-27 00:00:00
CM: 2026-04-28 00:00:00 2026-05-30 00:00:00
Row date: 2026-05-20 00:00:00


In [12]:
print(LST.columns.tolist())

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 14, Finished, Available, Finished, False)

['Contract', 'Assignment Date', 'Revoaktion', 'Assigned Agency', 'CD', 'Due Date', 'IBC Code', 'Batch']


## Importing Cash staging

In [15]:



# Define Fabric Lakehouse Output path
file_path = "/lakehouse/default/Files/Silver_Output/Cash_Payment_Breakup.xlsx"

# Read Excel into Cash dataframe
Cash = pd.read_excel(
    file_path,
    sheet_name="Sheet1",
    engine='openpyxl'
)



# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 18, Finished, Available, Finished, False)

In [16]:
value_date_cols = [col for col in Cash.columns if col.startswith("Value Date")]

Cash[value_date_cols] = (
    Cash[value_date_cols]
    .apply(pd.to_datetime, errors="coerce")
    .apply(lambda x: x.dt.normalize())
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 19, Finished, Available, Finished, False)

## Importing Billing Staging

### LM Bill1

In [17]:

import pandas as pd



# Define Fabric Output path
file_path = "/lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx"

# Read sheet named "BatchLM_Current_Bill"
BatchLM_Current_Bill = pd.read_excel(
    file_path,
    sheet_name=BatchLM_Current_Bill,   # ✅ string name of sheet
    engine='openpyxl'
)




# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)

# Rename column
BatchLM_Current_Bill = BatchLM_Current_Bill.rename(columns={"Cycle Day": "CD"})

# Convert data types
BatchLM_Current_Bill["CD"] = pd.to_numeric(
    BatchLM_Current_Bill["CD"], errors="coerce"
).astype("Int64")

BatchLM_Current_Bill["Net Amount"] = pd.to_numeric(
    BatchLM_Current_Bill["Net Amount"], errors="coerce"
).astype(float)

BatchLM_Current_Bill["Contract Number"] = pd.to_numeric(
    BatchLM_Current_Bill["Contract Number"], errors="coerce"
).astype("Int64")




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 20, Finished, Available, Finished, False)

In [18]:
BatchLM_Current_Bill = (
    BatchLM_Current_Bill.groupby("Contract Number", as_index=False)
      .agg({
          "Due Month": "first",
          "Units Billed": "sum",
          "Net Amount": "sum"
      })
)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 21, Finished, Available, Finished, False)

### LM Bill2

In [19]:


# Define Fabric Output path
file_path = "/lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx"

# Read sheet named "BatchLM_Next_Bill"
BatchLM_Next_Bill = pd.read_excel(
    file_path,
    sheet_name=BatchLM_Next_Bill,   # ✅ string name of sheet
    engine='openpyxl'
)



# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)

# Rename column
BatchLM_Next_Bill = BatchLM_Next_Bill.rename(columns={"Cycle Day": "CD"})

# Convert data types
BatchLM_Next_Bill["CD"] = pd.to_numeric(
    BatchLM_Next_Bill["CD"], errors="coerce"
).astype("Int64")

BatchLM_Next_Bill["Net Amount"] = pd.to_numeric(
    BatchLM_Next_Bill["Net Amount"], errors="coerce"
).astype(float)

BatchLM_Next_Bill["Contract Number"] = pd.to_numeric(
    BatchLM_Next_Bill["Contract Number"], errors="coerce"
).astype("Int64")




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 22, Finished, Available, Finished, False)

In [20]:
BatchLM_Next_Bill = (
    BatchLM_Next_Bill.groupby("Contract Number", as_index=False)
      .agg({
          "Due Month": "first",
          "Units Billed": "sum",
          "Net Amount": "sum"
      })
)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 23, Finished, Available, Finished, False)

### CM Bill1

In [22]:
# Importing CM Billing

# Define Fabric Output path
file_path = "/lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx"

# Read specific sheet named "BatchCM_Current_Bill"
BatchCM_Current_Bill = pd.read_excel(
    file_path,
    sheet_name=BatchCM_Current_Bill,   # ✅ STRING (sheet name)
    engine='openpyxl'
)


# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)

# Rename column
BatchCM_Current_Bill = BatchCM_Current_Bill.rename(columns={"Cycle Day": "CD"})

# Convert data types
BatchCM_Current_Bill["CD"] = pd.to_numeric(
    BatchCM_Current_Bill["CD"], errors="coerce"
).astype("Int64")

BatchCM_Current_Bill["Net Amount"] = pd.to_numeric(
    BatchCM_Current_Bill["Net Amount"], errors="coerce"
).astype(float)

BatchCM_Current_Bill["Contract Number"] = pd.to_numeric(
    BatchCM_Current_Bill["Contract Number"], errors="coerce"
).astype("Int64")




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 26, Finished, Available, Finished, False)

In [23]:
BatchCM_Current_Bill = (
    BatchCM_Current_Bill.groupby("Contract Number", as_index=False)
      .agg({
          "Due Month": "first",
          "Units Billed": "sum",
          "Net Amount": "sum"
      })
)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 27, Finished, Available, Finished, False)

### CM Bill2

In [25]:
# Importing CM Billing

import pandas as pd

# Fabric path
file_path = "/lakehouse/default/Files/Silver_Output/Due_Month_Report.xlsx"

# Read specific sheet (give actual sheet name as string)
BatchCM_Next_Bill = pd.read_excel(
    file_path,
    sheet_name=BatchCM_Next_Bill,   # ✅ must be STRING
    engine='openpyxl'
)


# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)
# Rename column
BatchCM_Next_Bill = BatchCM_Next_Bill.rename(columns={"Cycle Day": "CD"})

# Convert data types
BatchCM_Next_Bill["CD"] = pd.to_numeric(
    BatchCM_Next_Bill["CD"], errors="coerce"
).astype("Int64")

BatchCM_Next_Bill["Net Amount"] = pd.to_numeric(
    BatchCM_Next_Bill["Net Amount"], errors="coerce"
).astype(float)

BatchCM_Next_Bill["Contract Number"] = pd.to_numeric(
    BatchCM_Next_Bill["Contract Number"], errors="coerce"
).astype("Int64")



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 29, Finished, Available, Finished, False)

In [26]:
BatchCM_Next_Bill = (
    BatchCM_Next_Bill.groupby("Contract Number", as_index=False)
      .agg({
          "Due Month": "first",
          "Units Billed": "sum",
          "Net Amount": "sum"
      })
)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 30, Finished, Available, Finished, False)

# Importing RAW Billing

In [27]:
import pandas as pd

# Define Fabric Lakehouse path
file_path = "/lakehouse/default/Files/Raw_Billing.xlsx"

# Read specific sheet
Raw_Billing = pd.read_excel(
    file_path,
    sheet_name="Sheet1",
    engine='openpyxl'
)




# Concatenate all sheets vertically
#Bill1_Schedule = pd.concat(all_sheets.values(), ignore_index=True)

Raw_Billing = Raw_Billing.rename(columns={"Cycle Day": "CD"})

# Convert data types
Raw_Billing["CD"] = pd.to_numeric(
    Raw_Billing["CD"], errors="coerce"
).astype("Int64")

Raw_Billing["Net Amount"] = pd.to_numeric(
    Raw_Billing["Net Amount"], errors="coerce"
).astype(float)


Raw_Billing["Contract Number"] = pd.to_numeric(
    Raw_Billing["Contract Number"], errors="coerce"
).astype("Int64")

# Preview resul# Convert to datetime (keeps NaT if invalid)
Raw_Billing["Posting Date"]   = pd.to_datetime(Raw_Billing["Posting date"], errors="coerce").dt.normalize()
Raw_Billing["Due Date"]   = pd.to_datetime(Raw_Billing["Due Date (Print Doc)"], errors="coerce").dt.normalize()



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 31, Finished, Available, Finished, False)

In [28]:
Raw_Billing.columns

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 32, Finished, Available, Finished, False)

Index(['Contract Number', 'Calendar Year/Month', 'CD', 'Posting date',
       'BCM/Reversal Reason', 'Due Date (Print Doc)', 'Units Billed',
       'Net Amount', 'Status', 'Posting Date', 'Due Date'],
      dtype='object')

# Snapshots

In [29]:
LST_snapshot = LST.copy()
Bill1_Schedule_snapshot = Bill1_Schedule.copy()
Bill2_Schedule_snapshot = Bill2_Schedule.copy()
Cash_snapshot= Cash.copy()
Raw_Billing_snapshot= Raw_Billing.copy()


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 33, Finished, Available, Finished, False)

In [30]:
LST= LST_snapshot.copy()

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 34, Finished, Available, Finished, False)

# Generating Main File

## Picking Due Date from Bill1 Schedule

In [31]:
# Renmaing LST cycle day to CD
LST = LST.rename(columns={"Cycle Day": "CD"})


# Renaming columns in Bill1 schedule
Bill1_Schedule = Bill1_Schedule.rename(columns={"Cycle Day's": "CD"})
Bill1_Schedule = Bill1_Schedule.rename(columns={"Due Date": "Bill1_Sch_Due_Date"})


# Cathing Due Dates from BIll1 Schedule
LST = LST.merge(
    Bill1_Schedule[["CD", "Batch", "Bill1_Sch_Due_Date"]],
    on=["CD", "Batch"],
    how="left",
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 35, Finished, Available, Finished, False)

## Comparing Due date with Billing schedule Due Date

In [32]:
LST["Due Diff"] = (LST["Bill1_Sch_Due_Date"] - LST["Due Date"]).dt.days

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 36, Finished, Available, Finished, False)

## Compare Due Dates

In [33]:
LST["Bill1"] = pd.Series(dtype="Int64")
LST["Bill2"] = pd.Series(dtype="Int64")

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 37, Finished, Available, Finished, False)

## Bill1 General Fillup

In [34]:
# Create lookup dictionaries
lm_map = BatchLM_Current_Bill.set_index("Contract Number")["Net Amount"]
cm_map = BatchCM_Current_Bill.set_index("Contract Number")["Net Amount"]

#Fill Bill1 with conditions

LST["Bill1"] = np.where(
    (LST["Due Diff"] == 0) & (LST["Batch"] == BatchLM),
    LST["Contract"].map(lm_map),
    np.where(
        (LST["Due Diff"] == 0) & (LST["Batch"] == BatchCM),
        LST["Contract"].map(cm_map),
        pd.NA
    )
)



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 38, Finished, Available, Finished, False)

## Bill2 General Fillup

In [35]:



# Create lookup dictionaries
lm_map = BatchLM_Next_Bill.set_index("Contract Number")["Net Amount"]
cm_map = BatchCM_Next_Bill.set_index("Contract Number")["Net Amount"]

#Fill Bill2 with conditions
LST["Bill2"] = np.where(
    LST["Due Diff"] == 0,
    np.where(
        LST["Batch"] == BatchLM,
        LST["Contract"].map(lm_map),
        np.where(
            LST["Batch"] == BatchCM,
            LST["Contract"].map(cm_map),
            pd.NA
        )
    ),
    pd.NA
)







StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 39, Finished, Available, Finished, False)

## SUMIF Fill up

In [36]:
Raw_Billing["Net Amount"] = (
    Raw_Billing["Net Amount"]
    .astype(str)
    .str.replace(",", "", regex=False)      # remove thousand separators
    .str.replace("\u00A0", "", regex=False) # remove non-breaking spaces
    .str.strip()
)

Raw_Billing["Net Amount"] = pd.to_numeric(
    Raw_Billing["Net Amount"], errors="coerce"
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 40, Finished, Available, Finished, False)

In [37]:
LST['Bill1_orig'] = LST['Bill1']
LST['Bill2_orig'] = LST['Bill2']

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 41, Finished, Available, Finished, False)

In [38]:
# Ensure empty cells are treated as NaN
LST['Bill1'] = LST['Bill1'].replace('', pd.NA)
LST['Bill2'] = LST['Bill2'].replace('', pd.NA)

#Create a lookup from Raw_Billing
bill_map = (
    Raw_Billing
    .groupby(["Contract Number", "Due Date"])["Net Amount"]
    .sum()
)

#Apply condition on LST
mask = LST["Due Diff"] != 0

LST.loc[mask, "Bill1"] = (
    LST.loc[mask]
    .set_index(["Contract", "Due Date"])
    .index.map(bill_map)
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 42, Finished, Available, Finished, False)

In [39]:
# Filter rows where Due Diff != 0
mask = LST["Due Diff"] != 0

# Sum Bill1 for those rows
total_bill1_diff = int(LST.loc[mask, "Bill1"].sum())



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 43, Finished, Available, Finished, False)

In [40]:
# Filter rows where Due Diff != 0 AND Batch = "Mar'26"
mask = (LST["Due Diff"] != 0) & (LST["Batch"] == "Mar'26")

# Sum Bill1 for those rows
total_bill1_diff = int(LST.loc[mask, "Bill2"].sum())



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 44, Finished, Available, Finished, False)

## Bill2 Fillup with SUMIF

In [41]:
## Raw Billing Duplicate Remova
# Count total rows before removing duplicates
total_before = len(Raw_Billing)

# Remove duplicates based on specific columns
Raw_Billing = Raw_Billing.drop_duplicates(
    subset=["Contract Number", "Posting date", "Net Amount", "Due Date"],
    keep="first"  # Keep the first occurrence, drop others
)

# Count total rows after removing duplicates
total_after = len(Raw_Billing)

# Calculate how many duplicates were removed
duplicates_removed = total_before - total_after

print(f"Number of duplicate rows removed: {duplicates_removed}")

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 45, Finished, Available, Finished, False)

Number of duplicate rows removed: 88


In [42]:
Raw_Billing["Net Amount"] = (
    Raw_Billing["Net Amount"]
    .astype(str)
    .str.replace(",", "", regex=False)      # remove thousand separators
    .str.replace("\u00A0", "", regex=False) # remove non-breaking spaces
    .str.strip()
)

Raw_Billing["Net Amount"] = pd.to_numeric(
    Raw_Billing["Net Amount"], errors="coerce"
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 46, Finished, Available, Finished, False)

In [43]:
# Sort by Due Date
LST = LST.sort_values(by="Due Date")
# Sort by Contract
LST = LST.sort_values(by="Contract")


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 47, Finished, Available, Finished, False)

In [44]:

# Sort by Due Date first, then Contract Number
Raw_Billing = Raw_Billing.sort_values(
    ["Due Date", "Contract Number"]
)


"""""
This logic modified after last run to match e
# Shift Net Amount within each contract (now ordered by Due Date)
Raw_Billing["Next_Net_Amount"] = (
    Raw_Billing.groupby("Contract Number")["Net Amount"].shift(-1)
)
"""""


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 48, Finished, Available, Finished, False)

'""\nThis logic modified after last run to match e\n# Shift Net Amount within each contract (now ordered by Due Date)\nRaw_Billing["Next_Net_Amount"] = (\n    Raw_Billing.groupby("Contract Number")["Net Amount"].shift(-1)\n)\n'

In [45]:
# Step 1: Group duplicates (Contract + Due Date) so each date appears once
contract_summary = (
    Raw_Billing.groupby(["Contract Number", "Due Date"], as_index=False)["Net Amount"]
    .sum()
)

# Step 2: Sort properly
contract_summary = contract_summary.sort_values(["Contract Number", "Due Date"])

# Step 3: Shift only on unique due dates
contract_summary["Next_Net_Amount"] = (
    contract_summary.groupby("Contract Number")["Net Amount"].shift(-1)
)

# Step 4: Merge back if you need the next amount on original Raw_Billing
Raw_Billing = Raw_Billing.merge(
    contract_summary[
        ["Contract Number", "Due Date", "Next_Net_Amount"]
    ],
    on=["Contract Number", "Due Date"],
    how="left"
)



# Convert Next_Net_Amount to string for cleaning
Raw_Billing["Next_Net_Amount"] = (
    Raw_Billing["Next_Net_Amount"]
    .astype(str)
    .str.replace("#CALC!", "0", regex=False)
    .str.replace("#VALUE!", "0", regex=False)
    .str.replace("#DIV/0!", "0", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convert to numeric + replace invalid entries with 0
Raw_Billing["Next_Net_Amount"] = pd.to_numeric(
    Raw_Billing["Next_Net_Amount"], errors="coerce"
).fillna(0)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 49, Finished, Available, Finished, False)

In [46]:
# Replace 101 with the contract number you want to inspect
contract_number = 30086926

# Filter rows for that contract
contract_entries = Raw_Billing[Raw_Billing["Contract Number"] == contract_number]



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 50, Finished, Available, Finished, False)

In [47]:
# Ensure Bill2 exists and is float

LST["Bill2"] = LST["Bill2"].replace({pd.NA: np.nan}).astype(float)

#Create a lookup from Raw_Billing
bill_map = (
    Raw_Billing
    .groupby(["Contract Number", "Due Date"])["Next_Net_Amount"]
    .first()
)

#Apply condition on LST
mask = LST["Due Diff"] != 0

LST.loc[mask, "Bill2"] = (
    LST.loc[mask]
    .set_index(["Contract", "Due Date"])
    .index.map(bill_map)
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 51, Finished, Available, Finished, False)

In [48]:
# Filter rows where Due Diff != 0
mask = LST["Due Diff"] != 0

# Sum Bill1 for those rows
total_bill1_diff = int(LST.loc[mask, "Bill2"].sum())



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 52, Finished, Available, Finished, False)

## Merging Cash staging into LST

In [49]:
Cash = Cash.rename(columns={"Contract Number": "Contract"})
LST = LST.merge(
    Cash,
    on="Contract",
    how="left",
    suffixes=("", "_CASH")
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 53, Finished, Available, Finished, False)

# Calculation Total Cash

In [50]:
# Identify payment value Pairs


# Get all payment columns
payment_cols = [col for col in LST.columns if col.startswith("Payment Amount")]

# Get all corresponding Value Date columns
value_date_cols = [col for col in LST.columns if col.startswith("Value Date")]





StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 54, Finished, Available, Finished, False)

In [51]:
# Convert Dates for safety

LST["Assignment Date"] = pd.to_datetime(LST["Assignment Date"], errors="coerce").dt.normalize()
LST["Revoaktion"]      = pd.to_datetime(LST["Revoaktion"], errors="coerce").dt.normalize()

for col in value_date_cols:
    LST[col] = pd.to_datetime(LST[col], errors="coerce")

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 55, Finished, Available, Finished, False)

In [52]:
# Converting payments into numeric first

for p_col in payment_cols:
    LST[p_col] = (
        LST[p_col]
        .astype(str)
        .str.replace(",", "", regex=False)   # remove thousand separators
        .str.strip()
    )
    LST[p_col] = pd.to_numeric(LST[p_col], errors="coerce").fillna(0)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 56, Finished, Available, Finished, False)

In [53]:
# # 1. Clean payment columns (convert to numeric)
for p_col in payment_cols:
    LST[p_col] = (
        LST[p_col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    LST[p_col] = pd.to_numeric(LST[p_col], errors="coerce").fillna(0)

# 2. Initialize Total Cash
LST["Total Cash"] = 0.0   # float to avoid int issues

# 3. Loop and compute
for p_col, d_col in zip(payment_cols, value_date_cols):

    condition = (
        (LST[d_col] >= LST["Assignment Date"]) &
        (LST[d_col] <= LST["Revoaktion"])
    )

    LST["Total Cash"] += np.where(condition, LST[p_col], 0)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 57, Finished, Available, Finished, False)

# Calculating Total Billing

## Picking Bill2 Issue Date from Bill2 Schedule

In [54]:
Bill2_Schedule.columns = Bill2_Schedule.columns.str.replace("\n", " ", regex=True)
Bill2_Schedule.columns = Bill2_Schedule.columns.str.strip()
Bill2_Schedule.columns = Bill2_Schedule.columns.str.replace(" +", " ", regex=True)
Bill2_Schedule = Bill2_Schedule.rename(columns={"Cycle Day's": "CD"})

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 58, Finished, Available, Finished, False)

In [55]:
print(Bill2_Schedule.columns.tolist())

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 59, Finished, Available, Finished, False)

['Batch', 'Key_CD', 'CD', 'Meter Reading Print Date', 'Meter Reading Date', 'Documents Received at IBC', 'Transfer To ITG', 'Bills Print File sent to Vendors', 'Issue Date', 'Due Date']


In [56]:
# I dropped one CD here

# Renaming columns in Bill1 schedule

Bill2_Schedule = Bill2_Schedule.rename(columns={"Issue Date": "Bill2_Issue_Date"})

# Keep only one CD column
Bill2_Schedule = Bill2_Schedule.loc[:,~Bill2_Schedule.columns.duplicated()]

# Ensure column names are clean
Bill2_Schedule = Bill2_Schedule.rename(columns={"CD": "CD"})

# Now merge safely
LST = LST.merge(
    Bill2_Schedule[["Batch", "CD", "Bill2_Issue_Date"]],
    on=["CD", "Batch"],
    how="left"
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 60, Finished, Available, Finished, False)

## Calculating Total Bill

In [57]:
#Identify all Payment Date & Payment Amount columns dynamically
value_date_cols = [col for col in LST.columns if col.startswith("Value Date")]
payment_cols = [col for col in LST.columns if col.startswith("Payment Amount")]

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 61, Finished, Available, Finished, False)

In [58]:
#Convert all Value Date columns to datetime
for col in value_date_cols:
    LST[col] = pd.to_datetime(LST[col], errors="coerce")

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 62, Finished, Available, Finished, False)

In [59]:
# Check if any one payemnt comes after Bill2 Issue Date

LST["Has_Payment_After_Bill2"] = LST.apply(
    lambda row: any(
        (
            pd.notna(row[col])                             # Payment exists
            and row["Bill2_Issue_Date"] < row["Revoaktion"]  # Bill2 must be before revokation
            and row["Assignment Date"] <= row[col] <= row["Revoaktion"]  # Payment between assignment & revokation
            and row[col] > row["Bill2_Issue_Date"]          # Payment AFTER Bill2
        )
        for col in value_date_cols
    ),
    axis=1
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 63, Finished, Available, Finished, False)

In [60]:
LST["Total Bill"] = (
    LST["Bill1"]
    + LST["Bill2"].fillna(0) * LST["Has_Payment_After_Bill2"].astype(int)
)

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 64, Finished, Available, Finished, False)

In [61]:

# Sum all Total Bill values
total_bill_sum = int(LST["Total Bill"].sum())



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 65, Finished, Available, Finished, False)

In [62]:
# Filter rows where Due Diff == 0
mask = LST["Due Diff"] == 0

# Sum Total Bill for those rows
total_bill_sum_due0 = int(LST.loc[mask, "Total Bill"].sum())



StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 66, Finished, Available, Finished, False)

In [63]:
total_rows = len(LST)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 67, Finished, Available, Finished, False)

In [64]:
# Finding billing snapshot

# Define batch value
batch_value = "Mar'26"  # adjust if needed

# Filter rows
filtered_df = LST[(LST['Batch'] == batch_value) & (LST['Due Diff'] == 0)]

# Sum Bill1 and Bill2 for these filtered rows
total_bill1 = filtered_df['Bill1'].sum(skipna=True)
total_bill2 = filtered_df['Bill2'].sum(skipna=True)

total_bill = filtered_df['Total Bill'].sum(skipna=True)

# Display nicely
print(f"📊 Total Bill1 (Batch={batch_value}, Due Diff=0): {total_bill1:,.2f}")
print(f"📊 Total Bill2 (Batch={batch_value}, Due Diff=0): {total_bill2:,.2f}")
print(f"📊 Total Bill1 (Batch={batch_value}, Due Diff=0): {total_bill:,.2f}")

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 68, Finished, Available, Finished, False)

📊 Total Bill1 (Batch=Mar'26, Due Diff=0): 0.00
📊 Total Bill2 (Batch=Mar'26, Due Diff=0): 0.00
📊 Total Bill1 (Batch=Mar'26, Due Diff=0): 0.00


# Marking Instances

In [65]:
"""
LST["Instance"] = (
    LST.groupby(["Contract", "Batch"])
       .cumcount() + 1
)

"""
# Sort properly
LST = LST.sort_values(["Contract", "Batch", "Assignment Date"])

# Compare with previous Revoaktion
LST["Prev_Revo"] = LST.groupby(["Contract", "Batch"])["Revoaktion"].shift()

# Flag where reset is needed
LST["Reset_Flag"] = LST["Assignment Date"] > LST["Prev_Revo"]

# Create cumulative reset segments
LST["Reset_Group"] = LST.groupby(["Contract", "Batch"])["Reset_Flag"].cumsum()

# Assign instance numbers within each segment
LST["Instance"] = (
    LST.groupby(["Contract", "Batch", "Reset_Group"]).cumcount() + 1
)


StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 69, Finished, Available, Finished, False)

In [66]:
# My CheatSheet


#Raw_Billing.info()

LST.tail(50)
LST["Due Diff"].unique()

total_bill1 = LST["Total Bill"].sum()
total_bill1
#print("{:,}".format(total_bill1))
LST.head()


LST[LST["Contract"].astype(str) == "30086926"].T


total_bill = LST.loc[
    (LST["Batch"] == "Feb'26") & (LST["Instance"] == 1),
    "Total Bill"
].sum()
#print(f"{total_bill:,.0f}")



total_bill1 = LST["Bill1"].sum()
total_bill1
print(f"{total_bill:,.0f}")




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 70, Finished, Available, Finished, False)

0


In [67]:
last_date = LST['Assignment Date'].max()
last_date

StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 71, Finished, Available, Finished, False)

Timestamp('2026-05-22 00:00:00')

# Output

In [68]:
from datetime import datetime

# 1. Capture current date and time
now = datetime.now()

# 2. Format date & time
date_str = now.strftime("%d%b_%y")
time_str = now.strftime("%H%M")

# 3. Define Fabric Output path
file_path = f"/lakehouse/default/Files/Gold_Output/Output_RO_Report_{date_str}_{time_str}.xlsx"

# 4. Export dataframe
LST.to_excel(file_path, index=False)




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 72, Finished, Available, Finished, False)

#

In [69]:
# My CheatSheet


#Raw_Billing.info()

LST.tail(50)
LST["Due Diff"].unique()

total_bill1 = BatchCM_Current_Bill["Net Amount"].sum()
total_bill1
#print("{:,}".format(total_bill1))
LST.head()


LST[LST["Contract"].astype(str) == "32987551"].T

"""
total_bill = LST.loc[
    (LST["Batch"] == "Feb'26") & (LST["Instance"] == 1),
    "Total Bill"
].sum()
"""
print(f"{total_bill1:,.0f}")




StatementMeta(, 772d9d6d-1002-41fd-acc4-74c03cd0afff, 73, Finished, Available, Finished, False)

2,661,048,941
